# Curate the maize yield dataset
One row represents one maize field in one season. Start with the shared DuckDB database and run the notebook from top to bottom using the project Python environment. The build reads both pass archives and writes the local deliverables to `data/curated/pamely_felix/`.

| Predictor block | Columns | Time scale |
|---|---:|---|
| Pamely's ten Sentinel-2 bands and NDVI | 132 | Twelve calendar months in the recorded season year |
| Felix's rainfall and temperature | 18 | Nine months indexed from the start of the source season calendar |
| Felix's topsoil and terrain | 10 | Static |
| Felix's canopy changes and coverage counts | 12 | Early, middle and late thirds of the source season calendar |

The full-year table supports retrospective analysis. Three additional tables follow the recorded-date forecast design at 90, 60 and 30 days before harvest. All four use the same list of 172 predictors. Survey year, source, location, dates, and quality flags are excluded from that list. Missing measurements stay missing for the modeling stage.

## 1. Open the shared release
Keep these three files together in the following layout. Run all cells using a Python 3.11+ Jupyter kernel with DuckDB, NumPy, pandas and PyArrow installed.

```text
data/team_data_full_v1.duckdb
notebook/curate_dataset.ipynb
scripts/curation_optics.py
```

The database supplies both raw satellite archives, the field-season labels, the season calendars and the environmental measurements. The optical helper implements Pamely's pixel quality and reflectance method. Steps 5 and 6 implement Felix's environmental and previous-year features directly in this notebook.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
import hashlib
import json
import sys

import duckdb
import numpy as np
import pandas as pd
import pyarrow

def find_project():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "Project"):
            if (candidate / "notebook/curate_dataset.ipynb").is_file():
                return candidate
    raise FileNotFoundError("Start the kernel inside the course or project directory.")

root = find_project()
notebook_dir = root / "notebook"
scripts_dir = root / "scripts"
database = root / "data/team_data_full_v1.duckdb"
if not database.is_file():
    raise FileNotFoundError(f"Place the shared team_data_full_v1.duckdb at {database}")
output = root / "data/curated/pamely_felix"
sys.path.insert(0, str(scripts_dir))
from curation_optics import BANDS, monthly_optics

def digest(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

db = duckdb.connect(str(database), read_only=True)
ID, TARGET = "field_season_id", "yield_t_ha"
bands = list(BANDS)
print("Input:", database.relative_to(root))
print("Output:", output.relative_to(root))

Input: data/team_data_full_v1.duckdb
Output: data/curated/pamely_felix


## 2. Select field-seasons and preserve source flags
Use the inclusive contest rule: maize with a finite nonnegative yield (t/ha) and coordinates inside WGS84 bounds. Recorded zero yields and field-seasons with sparse imagery remain eligible. The original keep flag identifies the subset used in the combined-feature comparison. Keeping it as metadata also makes it possible to see which sources contributed the additional rows. Coordinate bounds alone do not establish field-level location accuracy.

In [2]:
labels = db.sql("""
    SELECT field_season_id, release, country_iso3, data_set AS dataset,
           season_year AS harvest_year, crop, latitude, longitude, loc_key,
           source_observation_id, overlap_group_id, location_quality, label_status,
           keep AS in_combination_cohort, exclusion_reason AS inherited_exclusion_reason,
           yield_t_ha
    FROM labels WHERE crop = 'maize' ORDER BY field_season_id
""").df()
assert labels[ID].notna().all() and labels[ID].is_unique
target_ok = np.isfinite(labels[TARGET]) & labels[TARGET].ge(0)
coords_ok = labels.latitude.between(-90, 90) & labels.longitude.between(-180, 180)
fields = labels.loc[target_ok & coords_ok].set_index(ID).copy()
ids = fields.index
fields["source_dataset"] = fields.country_iso3 + "/" + fields.dataset
db.register("curation_keys", fields.reset_index()[["release", ID]])
print(f"Maize records: {len(labels):,} → eligible yield: {target_ok.sum():,} "
      f"→ eligible coordinates and yield: {len(ids):,}")
print(f"Original comparison subset: {fields.in_combination_cohort.sum():,} field-seasons")

Maize records: 58,539 → eligible yield: 56,352 → eligible coordinates and yield: 42,968
Original comparison subset: 36,232 field-seasons


## 3. Apply Pamely's quality rule to the raw satellite passes
Process the two raw schemas separately. Within the harvest calendar year, Pamely ranks candidate products per acquisition, checks all ten bands on their native grids and retains pixels with known reflectance offsets and scene class 4 or 5 (vegetation or bare soil). An acquisition qualifies when at least half of the expected field footprint is usable. The field reflectance for that acquisition is the median of its usable pixels.

The [optical helper](../scripts/curation_optics.py) processes each raw schema on its own and returns a monthly table keyed by field-season ID and calendar month. All eligible maize sources use the same footprint, product selection and pixel rules.

In [3]:
monthly = monthly_optics(database, fields.reset_index()[[ID, "harvest_year"]])
assert not monthly.duplicated([ID, "month"]).any() and monthly.month.between(1, 12).all()
print(f"{monthly.acquisitions.sum():,} retained acquisition records → {len(monthly):,} field-months")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

1,597,114 retained acquisition records → 443,064 field-months


## 4. Form the 132 optical predictors
Each column holds the monthly median across qualifying acquisitions. NDVI uses red and near-infrared reflectance from the same acquisition before taking the monthly median. For these columns, `m1` means January and `m12` means December.

In [4]:
signals = bands + ["ndvi"]
optical = monthly.pivot(index=ID, columns="month", values=signals).reindex(
    index=ids, columns=pd.MultiIndex.from_product([signals, range(1, 13)]))
optical.columns = [f"{band}_m{month}" for band, month in optical.columns]
print(f"{optical.shape[1]} columns; {optical.notna().any(axis=1).sum():,} fields with usable optics")

132 columns; 38,116 fields with usable optics


## 5. Add Felix's weather, soil and terrain measurements
Use CHIRPS monthly rainfall in mm and AgERA5 air temperature in °C. Weather `m1` is the first scheduled **season month**. For the seven SoilGrids properties use the 0–5 cm layer in the archive's native units. Elevation and local relief are in metres, and slope is in degrees. Apply the source validity flags and leave fields without observations in the table.

In [5]:
weather_long = db.sql("""
    SELECT field_season_id, season_month,
           CASE WHEN chirps_valid AND chirps_precip_mm >= 0 THEN chirps_precip_mm END AS rain_mm,
           CASE WHEN agera5_temperature_valid THEN agera5_temperature_2m_k - 273.15 END AS temp_c
    FROM extra_rain_chirps FULL JOIN extra_weather_agera5
         USING(release,field_season_id,season_month,year,month)
    JOIN curation_keys USING(release,field_season_id)
""").df()
assert weather_long.season_month.between(1, 9).all()
weather = weather_long.pivot(index=ID, columns="season_month", values=["rain_mm", "temp_c"])
weather = weather.reindex(index=ids, columns=pd.MultiIndex.from_product([["rain_mm", "temp_c"], range(1, 10)]))
weather.columns = [f"extra_wx_{v}_m{m}" for v, m in weather.columns]

soils = ["clay", "sand", "phh2o", "soc", "nitrogen", "bdod", "cec"]
soil_sql = ", ".join(f"CASE WHEN soilgrids_valid THEN soilgrids_{s}_0_5cm_mean END "
                     f"AS extra_soil_{s}_0_5cm" for s in soils)
terrain_sql = ", ".join(f"CASE WHEN terrain_valid THEN {v} END AS extra_{v}"
                        for v in ["elevation_m", "slope_deg", "relief_3x3_m"])
site = db.sql(f"""
    SELECT field_season_id, {soil_sql}, {terrain_sql}
    FROM curation_keys LEFT JOIN extra_soil USING(release,field_season_id)
                       LEFT JOIN extra_terrain USING(release,field_season_id)
""").df().set_index(ID)
assert site.index.is_unique
site = site.reindex(ids)
print(f"{weather.shape[1]} weather columns and {site.shape[1]} static columns")

18 weather columns and 10 static columns


## 6. Reproduce Felix's previous-year comparisons
The tested 40-column extras block includes 12 satellite-derived history columns. These retain Felix's original clear-pass medians for current and previous years. Pair observations at the same field in the same calendar month exactly one year apart. For each third of the scheduled season, average the differences in greenness (NDVI), moisture (NDMI) and red-edge response (NDRE). Each pair receives the smaller of the two clear-pass counts, capped at three. A known season calendar with no NDVI pairs has a count of zero. A field outside the season calendar has missing history features.

In [6]:
calendar_key = [ID, "season_month", "which_year"]
calendar = db.sql("""
    SELECT field_season_id, season_month, which_year, year, month
    FROM monthly_clearest JOIN curation_keys USING(release,field_season_id)
""").df().set_index(calendar_key)
medians = db.sql("""
    SELECT field_season_id, season_month, which_year, n_clear, b04, b05, b08, b8a, b11
    FROM monthly_median JOIN curation_keys USING(release,field_season_id) WHERE n_clear > 0
""").df().set_index(calendar_key)
assert calendar.index.is_unique and medians.index.is_unique
assert medians.index.isin(calendar.index).all()
dated = medians.join(calendar)
pairs = dated.xs("this_season", level="which_year").add_suffix("_now").join(
    dated.xs("previous_year", level="which_year").add_suffix("_prior"), how="inner", validate="one_to_one")
pairs = pairs.loc[pairs.month_now.eq(pairs.month_prior) & pairs.year_now.eq(pairs.year_prior + 1)].copy()

schedule = calendar.xs("this_season", level="which_year").reset_index()
counts = schedule.groupby(ID).season_month.transform("size")
assert schedule.season_month.ge(1).all() and schedule.season_month.le(counts).all()
phases = ["early", "mid", "late"]
schedule["phase"] = pd.cut((schedule.season_month - 1) / counts, [0, 1/3, 2/3, 1],
                           labels=phases, right=False)
pairs = pairs.join(schedule.set_index([ID, "season_month"])[["phase"]], validate="one_to_one")
weights = pairs[["n_clear_now", "n_clear_prior"]].min(axis=1).clip(upper=3)
group = [pairs.index.get_level_values(ID), pairs.phase]

def normalized_difference(a, b):
    return (a - b).div((a + b).where((a >= 0) & (b >= 0) & (a + b > 0)))

history_parts = {}
for index_name, (a, b) in {"ndvi": ("b08", "b04"), "ndmi": ("b08", "b11"), "ndre": ("b8a", "b05")}.items():
    delta = (normalized_difference(pairs[a + "_now"], pairs[b + "_now"]) -
             normalized_difference(pairs[a + "_prior"], pairs[b + "_prior"]))
    numerator = (weights * delta).groupby(group, observed=True).sum(min_count=1)
    denominator = weights.where(delta.notna()).groupby(group, observed=True).sum(min_count=1)
    history_parts["delta_" + index_name] = (numerator / denominator).unstack().reindex(columns=phases)
    if index_name == "ndvi":
        n_pairs = delta.groupby(group, observed=True).count().unstack()
        history_parts["paired_months"] = n_pairs.reindex(
            index=schedule[ID].unique(), columns=phases).fillna(0)
history = pd.concat(history_parts, axis=1).reindex(ids)
history.columns = [f"extra_{v}_{phase}" for v, phase in history.columns]
print(f"{len(pairs):,} matched field-months; {len(history.columns)} history columns")

214,620 matched field-months; 12 history columns


## 7. Assemble the full-year dataset
Join the predictors by field-season ID and retain recorded planting and harvest dates as metadata. Coverage measures the presence of at least one observation in each block. The explicit predictor list is the modeling contract. Fit imputation and scaling inside the training folds.

In [7]:
dates = db.sql("""
    SELECT field_season_id, planting_date, harvest_date FROM raw_v1.fields
    UNION ALL
    SELECT field_season_id, planting_date, harvest_date FROM raw_oaf5.fields
""").df().set_index(ID)
fields = fields.join(dates, how="left", validate="one_to_one")
for col in ["planting_date", "harvest_date"]:
    fields[col] = pd.to_datetime(fields[col])
blocks = {"optical": optical, "weather": weather, "soil_and_terrain": site, "history": history}
assert all(block.index.equals(ids) for block in blocks.values())
features = pd.concat(blocks.values(), axis=1, verify_integrity=True)
predictors = features.columns.tolist()
assert features.shape == (len(ids), 172)
assert not np.isinf(features.to_numpy(dtype=float)).any()
assert not set(fields.columns).intersection(predictors)
full = fields.join(features, validate="one_to_one")
coverage = pd.DataFrame({
    "fields": 1, "comparison_rows": fields.in_combination_cohort.astype(int),
    "optical": optical.notna().any(axis=1), "weather": weather.notna().any(axis=1),
    "soil_or_terrain": site.notna().any(axis=1), "history_delta": history.filter(like="delta_").notna().any(axis=1)
}, index=ids).groupby([fields.country_iso3, fields.dataset, fields.harvest_year]).sum().astype(int)
print(coverage.groupby(level="country_iso3").sum().to_string())
print(f"Rows with all 172 predictors missing: {features.isna().all(axis=1).sum():,}")
print(f"Full-year table: {len(full):,} rows across {fields.source_dataset.nunique()} sources")

              fields  comparison_rows  optical  weather  soil_or_terrain  history_delta
country_iso3                                                                           
BDI             1556             1052     1052     1052             1052           1052
KEN            23549            23159    23158    23159            23159          22131
MWI             2534             2534     2534     2534             2534           2534
NGA              115                0        0        0                0              0
RWA             2854             1250     1253     1250             1250            847
TZA             4043             1975     1975     1975             1975           1975
UGA              246              246      246      246              246            246
ZMB             1470             1297     1297     1297             1297           1297
ZWE             6601             4719     6601     4719             4719           4719
Rows with all 172 predictors mis

## 8. Build the three pre-harvest views
Use a common subset of the comparison cohort so the same fields appear at every horizon. Require both recorded dates, a season of at most 550 days, a harvest in the label year, and a 90-day issue date after planting in that year. These issue dates use the realized harvest date and assume the measurements are available at month end.

For optics and weather, expose only whole calendar months between planting and issue. For Felix's history summaries, expose a phase only once all its scheduled months have known dates and the phase has ended. Static features are available at all horizons.

In [8]:
planting, harvest = fields.planting_date, fields.harvest_date
issue90 = harvest - pd.Timedelta(days=90)
eligible = (fields.in_combination_cohort & harvest.dt.year.eq(fields.harvest_year)
            & (harvest - planting).dt.days.between(1, 550)
            & issue90.gt(planting) & issue90.dt.year.eq(fields.harvest_year))
forecast_ids = ids[eligible]
dated_fields = fields.loc[forecast_ids]
schedule["start"] = pd.to_datetime(schedule[["year", "month"]].assign(day=1))
schedule["end"] = schedule.start + pd.offsets.MonthEnd(0)
season_starts = schedule.pivot(index=ID, columns="season_month", values="start").reindex(forecast_ids)
phase_bounds = schedule.groupby([ID, "phase"], observed=True).end.agg(["max", "count", "size"])
phase_ends = phase_bounds["max"].where(phase_bounds["count"].eq(phase_bounds["size"])).unstack().reindex(forecast_ids)

def make_forecast(lead_days):
    cutoff = (dated_fields.harvest_date - pd.Timedelta(days=lead_days)).rename("issue_date")
    x = features.loc[forecast_ids].copy()
    def month_is_available(start):
        return start.ge(dated_fields.planting_date) & (start + pd.offsets.MonthEnd(0)).le(cutoff)
    for month in range(1, 13):
        start = pd.to_datetime(pd.DataFrame({"year": dated_fields.harvest_year, "month": month, "day": 1}))
        x.loc[~month_is_available(start), [f"{b}_m{month}" for b in signals]] = np.nan
    for month in range(1, 10):
        x.loc[~month_is_available(season_starts[month]),
              [f"extra_wx_rain_mm_m{month}", f"extra_wx_temp_c_m{month}"]] = np.nan
    for phase in phases:
        phase_columns = [c for c in history if c.endswith("_" + phase)]
        x.loc[~phase_ends[phase].le(cutoff), phase_columns] = np.nan
    return pd.concat([dated_fields, cutoff, x], axis=1)

print(f"{len(forecast_ids):,} field-seasons meet the common forecast date rule")

21,103 field-seasons meet the common forecast date rule


## 9. Validate and publish the tables
The build checks every saved table against its in-memory values before publishing the directory. The manifest records the exact column order, source and method fingerprints, row counts and file hashes. The coverage CSV gives counts by source and harvest year. Rerunning against the same data checks the existing output. Choose a fresh output directory in step 1 to publish a changed release.

In [9]:
def save_table(frame, path):
    assert frame.index.is_unique and frame.index.name == ID
    export = frame.reset_index()
    export.to_parquet(path, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(path), export, check_exact=True)
    print(f"{path.name}: {len(frame):,} rows × {len(export.columns)} columns")

readme_lines = db.sql("SELECT text FROM readme ORDER BY line_no").df().text.to_list()
notices = readme_lines[readme_lines.index("SOURCES (keep these notices)") + 1:]
db.close()
output.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(dir=output.parent) as temp:
    staging = Path(temp) / "release"
    staging.mkdir()
    save_table(full, staging / "full_year.parquet")
    for lead in [90, 60, 30]:
        frame = make_forecast(lead)
        assert frame.index.equals(forecast_ids)
        save_table(frame, staging / f"forecast_{lead}d.parquet")
    coverage.to_csv(staging / "coverage_by_source_year.csv")
    (staging / "source_notices.txt").write_text("\n".join(notices) + "\n")
    notebook_cells = json.loads((notebook_dir / "curate_dataset.ipynb").read_text())["cells"]
    code_sources = ["".join(cell["source"]) for cell in notebook_cells if cell["cell_type"] == "code"]
    build = {
        "notebook": "notebook/curate_dataset.ipynb",
        "code_cells_sha256": hashlib.sha256(json.dumps(code_sources).encode()).hexdigest(),
        "helper": "scripts/curation_optics.py",
        "helper_sha256": digest(scripts_dir / "curation_optics.py"),
    }
    manifest = {
        "source": str(database.relative_to(root)), "source_sha256": digest(database),
        "build": build,
        "unit": "maize field-season", "id": ID, "target": TARGET, "target_unit": "t/ha",
        "predictors": predictors, "feature_blocks": {k: v.columns.tolist() for k, v in blocks.items()},
        "metadata": fields.columns.tolist(), "forecast_only_metadata": ["issue_date"],
        "rows": {"full_year": len(full), "in_combination_cohort": int(fields.in_combination_cohort.sum()),
                 "per_forecast_horizon": len(forecast_ids)},
        "optics": "Harvest-year calendar months; Pamely usable-pixel threshold >=50%.",
        "weather": "Scheduled season months 1-9; validity-filtered CHIRPS rain and AgERA5 temperature.",
        "history": "Felix weighted current-minus-previous-year differences and valid NDVI pair counts.",
        "forecast": "Recorded harvest minus 90/60/30 days; complete optical/weather months and completed phases.",
        "versions": {"pandas": pd.__version__, "numpy": np.__version__, "duckdb": duckdb.__version__,
                     "pyarrow": pyarrow.__version__},
        "file_sha256": {p.name: digest(p) for p in sorted(staging.iterdir())},
    }
    (staging / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
    if output.exists():
        previous = json.loads((output / "manifest.json").read_text())
        for name, checksum in previous["file_sha256"].items():
            if digest(output / name) != checksum:
                raise ValueError(f"Existing file failed its checksum: {name}")
        # Compression and floating-point reductions may differ across rebuilds.
        for key in ["source_sha256", "predictors", "feature_blocks", "metadata", "rows", "forecast"]:
            if previous[key] != manifest[key]:
                raise ValueError(f"Existing release differs in {key}. Choose a new output directory.")
        for path in staging.glob("*.parquet"):
            pd.testing.assert_frame_equal(pd.read_parquet(output / path.name), pd.read_parquet(path),
                                          check_exact=False, rtol=1e-10, atol=1e-12)
        for name in ["source_notices.txt", "coverage_by_source_year.csv"]:
            assert (output / name).read_bytes() == (staging / name).read_bytes()
        # Preserve original build provenance and hashes of the files on disk.
        # Record the independently verified build and canonical input location.
        previous.update(source=manifest["source"], verified_build=build)
        (staging / "manifest.json").write_text(json.dumps(previous, indent=2) + "\n")
        (staging / "manifest.json").replace(output / "manifest.json")
    else:
        staging.rename(output)
print("Published and checked:", output.relative_to(root))

full_year.parquet: 42,968 rows × 191 columns
forecast_90d.parquet: 21,103 rows × 192 columns
forecast_60d.parquet: 21,103 rows × 192 columns
forecast_30d.parquet: 21,103 rows × 192 columns
Published and checked: data/curated/pamely_felix


## 10. Load the dataset for modeling
Select a file by prediction time and use the ordered predictors in the manifest. To reproduce the original full-year comparison population, filter `full_year.parquet` on `in_combination_cohort`. All locations, dates and source identifiers remain available for defining splits and auditing coverage.

In [10]:
contract = json.loads((output / "manifest.json").read_text())
table = pd.read_parquet(output / "forecast_30d.parquet")  # Or full_year.parquet for retrospective work
X = table.loc[:, contract["predictors"]]
y = table[contract["target"]]
print(f"30-day modeling input: X {X.shape}, y {y.shape}")

30-day modeling input: X (21103, 172), y (21103,)
